In [ ]:
# -*- coding: utf-8 -*-
"""CNN_training.ipynb - Distance-3 CNN decoder (Path B & C)
Loads pre-generated CSVs and trains a simple CNN.
"""

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

print("Loading data...")
features_pretrain = pd.read_csv('features_pretrain.csv').values
labels_pretrain = pd.read_csv('labels_pretrain.csv').values.ravel()
features_finetune = pd.read_csv('features_finetune.csv').values
labels_finetune = pd.read_csv('labels_finetune.csv').values.ravel()

# Split into train/test (80/20) – same splits as before
X_train_pre, X_test_pre, y_train_pre, y_test_pre = train_test_split(
    features_pretrain, labels_pretrain, test_size=0.2, random_state=42
)
X_train_fine, X_test_fine, y_train_fine, y_test_fine = train_test_split(
    features_finetune, labels_finetune, test_size=0.2, random_state=42
)

# Convert to PyTorch tensors
train_dataset = TensorDataset(
    torch.tensor(X_train_pre, dtype=torch.float32),
    torch.tensor(y_train_pre, dtype=torch.float32).unsqueeze(1)
)
test_dataset = TensorDataset(
    torch.tensor(X_test_pre, dtype=torch.float32),
    torch.tensor(y_test_pre, dtype=torch.float32).unsqueeze(1)
)
train_fine_dataset = TensorDataset(
    torch.tensor(X_train_fine, dtype=torch.float32),
    torch.tensor(y_train_fine, dtype=torch.float32).unsqueeze(1)
)
test_fine_dataset = TensorDataset(
    torch.tensor(X_test_fine, dtype=torch.float32),
    torch.tensor(y_test_fine, dtype=torch.float32).unsqueeze(1)
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# ============================================================
# CNN Model Definition
# ============================================================
class SimpleCNN(nn.Module):
    """
    Takes input shape (batch, 5, 8) – treat as 2D grid with 1 channel.
    """
    def __init__(self):
        super().__init__()
        # Input: (batch, 1, 5, 8)
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)  # -> (16,5,8)
        self.pool = nn.MaxPool2d(2,2)  # -> (16,2,4) after pooling (5->2, 8->4)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1) # -> (32,2,4)
        # After second conv + pool -> (32,1,2)  (2/2=1, 4/2=2)
        # Flatten: 32*1*2 = 64
        self.fc1 = nn.Linear(32 * 1 * 2, 64)
        self.fc2 = nn.Linear(64, 1)

    def forward(self, x):
        # x: (batch, 40) -> reshape to (batch, 1, 5, 8)
        x = x.view(-1, 1, 5, 8)
        x = torch.relu(self.conv1(x))
        x = self.pool(x)
        x = torch.relu(self.conv2(x))
        x = self.pool(x)          # now (batch, 32, 1, 2)
        x = x.view(x.size(0), -1) # flatten
        x = torch.relu(self.fc1(x))
        x = torch.sigmoid(self.fc2(x))
        return x

# ============================================================
# Training function (reusable)
# ============================================================
def train_model(model, train_loader, test_loader, epochs, lr=0.001, patience=3):
    model.to(device)
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    best_loss = float('inf')
    patience_counter = 0

    for epoch in range(1, epochs+1):
        model.train()
        running_loss = 0.0
        for batch_x, batch_y in train_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            optimizer.zero_grad()
            preds = model(batch_x)
            loss = criterion(preds, batch_y)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * batch_x.size(0)
        train_loss = running_loss / len(train_loader.dataset)

        # Validation
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for batch_x, batch_y in test_loader:
                batch_x, batch_y = batch_x.to(device), batch_y.to(device)
                preds = model(batch_x)
                loss = criterion(preds, batch_y)
                val_loss += loss.item() * batch_x.size(0)
        val_loss = val_loss / len(test_loader.dataset)

        if epoch % 5 == 0 or epoch == 1:
            print(f"Epoch {epoch:02d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

        # Early stopping
        if val_loss < best_loss:
            best_loss = val_loss
            patience_counter = 0
            torch.save(model.state_dict(), 'best_cnn.pth')
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"Early stopping at epoch {epoch}")
                break
    return model

# ============================================================
# DataLoaders
# ============================================================
BATCH_SIZE = 128
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
train_loader_fine = DataLoader(train_fine_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader_fine = DataLoader(test_fine_dataset, batch_size=BATCH_SIZE, shuffle=False)

# ============================================================
# Path B: Pre-train on symmetric noise
# ============================================================
print("\n--- Path B: Training CNN on Symmetric Noise ---")
model_b = SimpleCNN()
train_model(model_b, train_loader, test_loader, epochs=50, lr=0.001)
model_b.load_state_dict(torch.load('best_cnn.pth'))  # load best
model_b.eval()

# Predict on symmetric test set
preds_b = []
with torch.no_grad():
    for batch_x, _ in test_loader:
        batch_x = batch_x.to(device)
        probs = model_b(batch_x)
        preds_b.extend((probs >= 0.5).int().cpu().numpy().flatten())
preds_b = np.array(preds_b)
ler_b = np.mean(preds_b != y_test_pre)
print(f"Path B (Symmetric) LER: {ler_b:.4f}")

# Save predictions
pd.DataFrame({'GroundTruth': y_test_pre, 'CNN_B_Prediction': preds_b}).to_csv('cnn_results_path_b.csv', index=False)

# ============================================================
# Path C: Pre-train on symmetric, then fine-tune on asymmetric
# ============================================================
print("\n--- Path C: Fine-tuning CNN on Asymmetric Noise (transfer learning) ---")
# Start from the pre-trained model (best from Path B)
model_c = SimpleCNN()
model_c.load_state_dict(torch.load('best_cnn.pth'))  # load pre-trained weights

# Fine-tune with lower learning rate
train_model(model_c, train_loader_fine, test_loader_fine, epochs=20, lr=0.0001, patience=3)
model_c.load_state_dict(torch.load('best_cnn.pth'))  # best from fine-tuning
model_c.eval()

# Predict on asymmetric test set
preds_c = []
with torch.no_grad():
    for batch_x, _ in test_loader_fine:
        batch_x = batch_x.to(device)
        probs = model_c(batch_x)
        preds_c.extend((probs >= 0.5).int().cpu().numpy().flatten())
preds_c = np.array(preds_c)
ler_c = np.mean(preds_c != y_test_fine)
print(f"Path C (Asymmetric) LER: {ler_c:.4f}")

# Save predictions
pd.DataFrame({'GroundTruth': y_test_fine, 'CNN_C_Prediction': preds_c}).to_csv('cnn_results_path_c.csv', index=False)

# ============================================================
# Summary
# ============================================================
print("\n" + "="*50)
print("CNN Training Complete")
print("="*50)
print(f"Path B LER (Symmetric): {ler_b:.4f}")
print(f"Path C LER (Asymmetric): {ler_c:.4f}")
print("\nSaved files:")
print("  - cnn_results_path_b.csv")
print("  - cnn_results_path_c.csv")
print("  - best_cnn.pth (latest best model)")

Loading data...
Device: cuda

--- Path B: Training CNN on Symmetric Noise ---
Epoch 01 | Train Loss: 0.4508 | Val Loss: 0.3939
Epoch 05 | Train Loss: 0.4016 | Val Loss: 0.3789
Epoch 10 | Train Loss: 0.3504 | Val Loss: 0.3448
Epoch 15 | Train Loss: 0.3160 | Val Loss: 0.3296
Epoch 20 | Train Loss: 0.2885 | Val Loss: 0.3137
Epoch 25 | Train Loss: 0.2564 | Val Loss: 0.3133
Early stopping at epoch 27
Path B (Symmetric) LER: 0.1115

--- Path C: Fine-tuning CNN on Asymmetric Noise (transfer learning) ---
Epoch 01 | Train Loss: 0.4156 | Val Loss: 0.4137
Epoch 05 | Train Loss: 0.4028 | Val Loss: 0.4068
Epoch 10 | Train Loss: 0.3960 | Val Loss: 0.4028
Epoch 15 | Train Loss: 0.3915 | Val Loss: 0.4014
Epoch 20 | Train Loss: 0.3875 | Val Loss: 0.3993
Path C (Asymmetric) LER: 0.1515

CNN Training Complete
Path B LER (Symmetric): 0.1115
Path C LER (Asymmetric): 0.1515

Saved files:
  - cnn_results_path_b.csv
  - cnn_results_path_c.csv
  - best_cnn.pth (latest best model)
